In [1]:
# 1. Install Dependencies
!pip install -U ultralytics wandb roboflow -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.3/27.3 MB 67.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.9/207.9 kB 16.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 60.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.5/5.5 MB 100.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.2/12.2 MB 71.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.29.0 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
dask-cuda 26.2.0 requires cuda-core==0.3.*, but you have cuda-core 1.0.1 which is incompatible.
dask-cuda 26

In [2]:
from roboflow import Roboflow
from kaggle_secrets import UserSecretsClient

user_secrets = UserSecretsClient()
roboflow_api_key = user_secrets.get_secret("ROBO_FLOW_API")

rf = Roboflow(api_key=roboflow_api_key)
project = rf.workspace("obeds-workspace-m260m").project("benchmarking-models-road-anom")
version = project.version(2)
dataset = version.download("yolov8")
                

loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Benchmarking-Models---Road-Anom.-2 in yolov8:: 100%|██████████| 8893/8893 [00:01<00:00, 7004.75it/s] 


Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [3]:
# 3. Authenticate Weights & Biases
#from kaggle_secrets import UserSecretsClient
import wandb

#user_secrets = UserSecretsClient()
wandb_api_key = user_secrets.get_secret("WANDB_API_KEY_DLI")
wandb.login(key=wandb_api_key)

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: [wandb.login()] Using explicit session credentials for https://api.wandb.ai.
wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: obedhonoureje (obedhonoureje-federal-university-of-technology-minna) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


True

In [4]:
#4 Train
import os
import shutil
import pandas as pd
import wandb
from ultralytics import YOLO
from kaggle_secrets import UserSecretsClient

# 1. Authenticate W&B
user_secrets = UserSecretsClient()
wandb.login(key=user_secrets.get_secret("WANDB_API_KEY_DLI"))

# Initialize run
run = wandb.init(project="road-anomaly-benchmark", name="yolov12n-baseline")

# 2. Train YOLOv12
print("\n--- Training YOLOv12n Baseline ---")
model = YOLO('yolo12n.pt')

# We explicitly set project to /kaggle/working/results to keep it clean
results = model.train(
    data=dataset.location + '/data.yaml', # Uses the Roboflow dataset path
    epochs=100, 
    imgsz=640, 
    batch=16,
    project="/kaggle/working/results",
    name="yolov12n-baseline"
)

# 3. Bulletproof W&B Logging (From your CityPersons script!)
print("\n--- Uploading Telemetry to WandB ---")
results_csv = "/kaggle/working/results/yolov12n-baseline/results.csv"

if os.path.exists(results_csv):
    df = pd.read_csv(results_csv)
    df.columns = [c.strip() for c in df.columns]
    
    # Dynamically find the mAP columns
    map50_col   = next((c for c in df.columns if "mAP50" in c and "95" not in c), None)
    map5095_col = next((c for c in df.columns if "mAP50-95" in c or "0.5:0.95" in c), None)
    
    # Stream the data to the W&B charts
    for i, row in df.iterrows():
        log_dict = {
            "epoch": i + 1,
            "val/mAP50": float(row[map50_col]),
            "val/mAP50-95": float(row[map5095_col]),
        }
        run.log(log_dict, step=i)

run.finish()

# 4. Rescue the Weights
print("\n--- Saving Weights ---")
source_weights = "/kaggle/working/results/yolov12n-baseline/weights/best.pt"
safe_weights = "/kaggle/working/yolov12n_best.pt"

if os.path.exists(source_weights):
    shutil.copy(source_weights, safe_weights)
    print(f"✓ Weights safely copied to: {safe_weights}")

print("✓ YOLOv12 Pipeline Complete!")

wandb: WARNING If you're specifying your api key in code, ensure this code is not shared publicly.
wandb: WARNING Consider setting the WANDB_API_KEY environment variable, or running `wandb login` from the command line.
wandb: WARNING [wandb.login()] Changing session credentials to explicit value for https://api.wandb.ai.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Tracking run with wandb version 0.27.0
wandb: Run data is saved locally in /kaggle/working/wandb/run-20260528_151335-vhlbzp1k
wandb: Run `wandb offline` to turn off syncing.
wandb: Syncing run yolov12n-baseline
wandb: ⭐️ View project at https://wandb.ai/obedhonoureje-federal-university-of-technology-minna/road-anomaly-benchmark
wandb: 🚀 View run at https://wandb.ai/obedhonoureje-federal-university-of-technology-minna/road-anomaly-benchmark/runs/vhlbzp1k



--- Training YOLOv12n Baseline ---
Ultralytics 8.4.56 🚀 Python-3.12.13 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/Benchmarking-Models---Road-Anom.-2/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolo12n.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=yolov12n-baseline, nbs=64, nms=Fal

wandb: updating run metadata



--- Uploading Telemetry to WandB ---


wandb: uploading history steps 0-99, summary, console lines 562-569
wandb: 
wandb: Run history:
wandb:        epoch ▁▁▁▁▁▂▂▂▂▂▃▃▃▃▃▃▃▃▄▄▄▄▄▅▅▅▆▆▆▆▆▆▆▇▇▇▇▇██
wandb:    val/mAP50 ▁▂▃▃▄▅▅▅▅▅▆▆▆▆▇▇▇▇▇▇▇▇▇▇▇▇▇█▇███████████
wandb: val/mAP50-95 ▁▂▃▃▄▅▄▅▅▅▅▅▅▆▆▆▆▆▆▆▇▆▇▇▇▇▇▇▇▇▇▇▇███████
wandb: 
wandb: Run summary:
wandb:        epoch 100
wandb:    val/mAP50 0.8952
wandb: val/mAP50-95 0.58743
wandb: 
wandb: 🚀 View run yolov12n-baseline at: https://wandb.ai/obedhonoureje-federal-university-of-technology-minna/road-anomaly-benchmark/runs/vhlbzp1k
wandb: ⭐️ View project at: https://wandb.ai/obedhonoureje-federal-university-of-technology-minna/road-anomaly-benchmark
wandb: Synced 5 W&B file(s), 0 media file(s), 0 artifact file(s) and 0 other file(s)
wandb: Find logs at: ./wandb/run-20260528_151335-vhlbzp1k/logs



--- Saving Weights ---
✓ Weights safely copied to: /kaggle/working/yolov12n_best.pt
✓ YOLOv12 Pipeline Complete!
